In [2]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [5]:
#@title load necessary packages
!pip install sentence_transformers
!pip install faiss-gpu -q
!pip install rapidfuzz

ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


In [8]:
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 69.8 MB/s eta 0:00:00


In [14]:
import pandas as pd
import numpy as np
import re
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz

### === STEP 1: LOAD DATA === ###
# Update these file paths as needed
large_df = pd.read_csv("/content/drive/MyDrive/PhD Data/07 Name matching/raw/USPTO disambiguated assignees.csv")  # 400K companies
small_df = pd.read_csv("/content/drive/MyDrive/PhD Data/07 Name matching/raw/ All deals target names.csv")  # 1.4K companies

# Ensure column names are correct
large_names = large_df.iloc[:, 0].astype(str).tolist()  # Assuming company names in the first column
small_names = small_df.iloc[:, 0].astype(str).tolist()

### === STEP 2: PREPROCESS COMPANY NAMES === ###
def clean_company_name(name):
    name = name.lower()
    name = re.sub(r'[^a-z0-9\s]', '', name)  # Remove special characters
    name = re.sub(r'\s+', ' ', name).strip()  # Remove extra spaces
    common_words = {"inc", "corp", "ltd", "llc", "co", "sa", "ag", "plc", "limited"}
    name = " ".join([word for word in name.split() if word not in common_words])  # Remove stopwords
    return name

# Apply preprocessing
large_names_clean = [name for name in large_names]
small_names_clean = [name for name in small_names]

### === METHOD 1: BERT + FAISS + LEVENSHTEIN === ###
def match_with_bert_faiss():
    print("Matching with BERT + FAISS + Levenshtein...")

    # Load BERT model (efficient transformer)
    model = SentenceTransformer("all-MiniLM-L6-v2")  # Optimized for speed

    # Convert company names into embeddings
    embeddings_large = model.encode(large_names_clean, convert_to_numpy=True)
    embeddings_small = model.encode(small_names_clean, convert_to_numpy=True)

    # Create FAISS index for fast nearest neighbor search
    d = embeddings_large.shape[1]  # Dimension of embeddings
    index = faiss.IndexFlatL2(d)
    index.add(embeddings_large)

    # Search for nearest match
    D, I = index.search(embeddings_small, 1)  # Get closest match

    # Extract best matches
    matched_names = [large_names[i] for i in I.flatten()]
    similarity_scores = [1 - d for d in D.flatten()]  # Convert distance to similarity

    # Apply Levenshtein Distance for typo correction
    final_matches = []
    for i, name in enumerate(small_names):
        bert_match = matched_names[i]
        levenshtein_score = fuzz.ratio(clean_company_name(name), clean_company_name(bert_match))

        # Accept match if BERT is highly confident OR if Levenshtein catches a typo
        if similarity_scores[i] > 0.8 or levenshtein_score > 80:
            final_matches.append((name, bert_match, max(similarity_scores[i], levenshtein_score / 100)))
        else:
            final_matches.append((name, "NO MATCH", 0))

    # Save results to CSV
    matched_df = pd.DataFrame(final_matches, columns=["small_dataset_name", "matched_large_dataset_name", "final_score"])
    matched_df.to_csv("matched_bert_faiss_levenshtein.csv", index=False)
    print("✅ BERT + FAISS + Levenshtein matching complete! Results saved as 'matched_bert_faiss_levenshtein.csv'.")

### === METHOD 2: TF-IDF + COSINE SIMILARITY === ###
def match_with_tfidf():
    print("Matching with TF-IDF + Cosine Similarity...")

    # TF-IDF Vectorization
    vectorizer = TfidfVectorizer()
    tfidf_matrix_large = vectorizer.fit_transform(large_names_clean)
    tfidf_matrix_small = vectorizer.transform(small_names_clean)

    # Compute Cosine Similarity
    similarities = cosine_similarity(tfidf_matrix_small, tfidf_matrix_large)

    # Find Best Matches
    matches = np.argmax(similarities, axis=1)
    similarity_scores = [similarities[j, i] for j, i in enumerate(matches)]

    # Create a DataFrame with Matches
    matched_df = pd.DataFrame({
        "small_dataset_name": small_names,
        "matched_large_dataset_name": [large_names[i] for i in matches],
        "similarity_score": similarity_scores
    })

    # Save results to CSV
    matched_df.to_csv("matched_tfidf_cosine.csv", index=False)
    print("✅ TF-IDF + Cosine Similarity matching complete! Results saved as 'matched_tfidf_cosine.csv'.")



In [15]:
### === RUN BOTH MATCHING METHODS === ###
if __name__ == "__main__":
    match_with_bert_faiss()
    match_with_tfidf()


Matching with BERT + FAISS + Levenshtein...
✅ BERT + FAISS + Levenshtein matching complete! Results saved as 'matched_bert_faiss_levenshtein.csv'.
Matching with TF-IDF + Cosine Similarity...
✅ TF-IDF + Cosine Similarity matching complete! Results saved as 'matched_tfidf_cosine.csv'.
